# Piilevän riskin löytäminen offshore-rakenteista

## ICIJ Paradise Papers -analyysi Jennerissä

Tämä muistikirja suorittaa päästä päähän petosanalytiikan putken
todellista ICIJ Paradise Papers -vuotoa vastaan — **163 435 solmua**,
jotka kattavat 24 957 offshore-yhteisöä, 77 012 toimihenkilöä,
59 228 osoitetta ja 2 031 välittäjää, yhdistettyinä 221 112
OFFICER_OF-suhteella.

Tietolähteenä on Jenner Workspace -alustan jaettu
`step-neo4j`-palvelu — Neo4j 5.26 Community Edition Graph Data
Science -laajennuksella, joka sisältää julkisen ICIJ Paradise
Papers -vedoksen palvelintasolla vain luku -tilassa. Työtilan podit
tavoittavat sen osoitteessa `bolt://step-neo4j:7687`
ympäristömuuttujien `JENNER_NEO4J_HOST` ja `JENNER_NEO4J_PASS`
kautta, jotka alusta paistaa jokaiseen työtilan podiin; erillistä
tenanttikohtaista asennusta ei tarvita. Jokainen tämän muistikirjan
solu suoritetaan tuota elävää graafia vastaan — putkessa ei ole
missään kohtaa synteettistä tai otokseen perustuvaa dataa.

Analyysi on jaettu viiteentoista osaan, jotka on kääritty yhteen
`ODS PDF` -direktiiviin, jotta kirjoitettu raportti sisältää koko
tarinan:

| Osa | Aihe |
|---|---|
| 1 | Yhteyden muodostus ja datan koon mittaus |
| 2 | Lainkäyttöalueiden jakauma |
| 3 | Louvain-yhteisöjen tunnistus |
| 4 | PageRank-keskeisyys |
| 5 | Yhteisökohtainen piirteiden muodostus |
| 6 | OFAC-SDN-seulonta |
| 7 | Kaplan-Meier-eloonjääminen |
| 8 | Coxin suhteellinen vaarantuvuus |
| 9 | Logistinen kompleksisuusregressio |
| 10 | Konsolidoidut avaintunnusluvut |
| 11 | Toimihenkilökeskeinen vaikutusvallan pisteytys |
| 12 | Ajalliset muodostumiskuviot |
| 13 | Vuotojen välinen vertailu |
| 14 | OpenSanctions-laajempi rikastus |
| 15 | Yhdistetty yhteisöjen riskipisteytys |

**Ensisijainen tietolähde:** International Consortium of
Investigative Journalists, *Paradise Papers* (2017). Julkinen vedos
saatavilla osoitteessa
<https://offshoreleaks.icij.org/pages/database>.

**Toissijainen `data/`-hakemistoon talletettu data:**

- `data/ofac_sdn.csv` — Yhdysvaltain OFAC:n Specially Designated
  Nationals -otos (500 riviä, huhtikuu 2026)
- `data/opensanctions_default.csv` — OpenSanctions-konsolidoitu
  pakotteiden tilannekuva, 2026-04-17
- `data/tax_haven_ranks.csv` — Tax Justice Networkin Corporate Tax
  Haven Index 2021 -julkaistut sijoitukset


## 1. Yhteyden muodostus ja datan koon mittaus

Avaamme `LIBNAME ... GRAPH ENGINE=NEO4J` -yhteyden jaettuun
tutkimuspalvelimeen. Ytimen ympäristöön on asetettu palvelin ja
salasana, joten SYSGET-haku ratkeaa automaattisesti.


In [3]:
/* Avaa yksi ODS PDF -kääre koko analyysin ympärille. Jokainen
   PROC-tuloste osasta 1 eteenpäin tallentuu tähän raporttiin.
   PDF suljetaan aivan muistikirjan lopussa, joten kirjoitettu
   raportti sisältää koko kertomuksen: mittakaava, lainkäyttöalueet,
   yhteisöt, PageRank, piirteet, pakotteet, eloonjääminen, Cox,
   logistinen, toimihenkilönäkymä, ajallinen, vuotojen välinen,
   laajemmat pakotteet ja lopullinen yhdistetty riskipisteytys. */
ods pdf tiedosto="output/icij_fraud_report.pdf" style=journal;

otsikko "ICIJ Paradise Papers — Hidden-Risk Analysis";


NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Ratkaise talletettujen demo-CSV-tiedostojen sijainti.
   Oletus: suhteellinen polku `data/` (ratkeaa, kun ytimen
   työhakemisto on muistikirjan hakemisto — Jupyterin vakiokäytös).
   Ohitus: aseta JENNER_ICIJ_DATA ytimen ympäristöön absoluuttiseksi
   poluksi, jos ydin käynnistetään eri työhakemistosta. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%pituus(%superq(_raw_icij_data))=0,
                              tiedot, %superq(_raw_icij_data)));
%kirjoita NOTE: ICIJ demo tiedot directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

proseduuri gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

proseduuri gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Näytä lukumäärät PROC MEANS SUM -komennolla (jokainen datajoukko
   on yksirivinen lukumäärä, joten SUM == arvo — antaa klassisen SAS-
   yhteenvetolaatikon ilman DATA _null_ PUT -kikkaa). */
proseduuri keskiarvot tiedot=node_total sum maxdec=0;
    muuttuja total_nodes;
    otsikko "Total nodes in the Paradise-Papers leaked graph";
suorita;

proseduuri keskiarvot tiedot=label_counts sum maxdec=0;
    muuttuja n_entity n_officer n_intermediary n_address;
    nimike n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    otsikko "Node counts by label";
suorita;


                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Missä raha on rekisteröity?

Paradise Papers -vuoto kattaa yhteisöjä, jotka on rekisteröity noin
50 lainkäyttöalueelle. Vaakapylväskaavio kymmenestä suurimmasta
lainkäyttöalueesta osoittaa, kuinka keskittynyttä offshore-toiminta
on kourallisessa verotuksellisesti edullisia alueita. Bermuda ja
Caymansaaret hallitsevat — yhteensä 18 204 yhteisöä (73 % nimetyistä
24 957:stä).


In [ ]:
proseduuri gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

proseduuri tulosta tiedot=jurisdictions nimike;
    otsikko "Top 10 Paradise-Papers Jurisdictions";
    nimike jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    muoto n_entity comma12.;
suorita;

/* Pareto-valmistelu: Cypher-kysely järjestää lainkäyttöalueet jo
   suuresta pieneen, joten kertaamme juoksevan summan ja ilmaisemme
   sen prosenttina kymmenen kärjen kokonaissummasta. Toissijaisen
   akselin viivapäällys nousee ensimmäisestä lainkäyttöalueesta
   100 %:iin kymmenennessä. */
proseduuri sql noprint;
    valitse sum(n_entity) into :_grand
    from jurisdictions;
quit;

tiedot jurisdictions_pareto;
    aseta jurisdictions;
    pidä _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    poista _cum;
suorita;

proseduuri sgplot tiedot=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis nimike="Jurisdiction (ISO-2)";
    yaxis nimike="Number of Entities";
    y2axis nimike="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    otsikko "Top 10 Paradise-Papers Jurisdictions — Pareto";
suorita;
otsikko;


## 3. Kuka ryhmittyy yhteen? Louvain-yhteisöjen tunnistus

`PROC NETWORK` ajaa Louvainin paikallisesti
`(Officer)-[OFFICER_OF]->(Entity)` -kaksijakoisella graafilla ja
noutaa 5 000 asteeltaan suurimman toimihenkilön aligraafin vain
luku -tyyppisellä Cypher-`MATCH`-kyselyllä `step-neo4j`:tä vastaan.
Alustan jaettu `step-neo4j` pakottaa asetuksen
`server.databases.default_to_read_only=true`, joten kaikki graafi-
analytiikka, joka muuttaisi tietokantaa (GDS:n `gds.graph.project`
-vaihe, jonka `PROC LINKS` olisi kutsunut), estetään palvelimella.
`PROC NETWORK` kiertää tämän — se lähettää täsmätyt rivit Boltin yli
ja ajaa algoritmin prosessin sisällä työtilan podissa.

Otamme otoksen 5 000 kytketyimmästä toimihenkilöstä, koska Louvain
koko kaksijakoisella graafilla (165 t reunaa) vie minuutteja
NetworkX:ssä, kun taas Java-GDS tekee sen sekunneissa; demon
interaktiivisen rytmin kannalta aligraafi säilyttää analyyttisen
pääviestin (suurivolyymisten välittäjien yhteisörakenne) ja pitää
ajoajan nopeana.

Sen jälkeen liitämme yhteisötunnisteet takaisin yhteisötauluun,
jotta jatko-osat voivat mitata rakenteellisten ryppäiden koon.


In [ ]:
proseduuri network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id asti=b_node_id;
    community algorithm=louvain;
suorita;

/* Nimeä PROC NETWORKin `community_1`-sarake uudelleen
   `community_id`-nimeksi, jota jatkossa oleva PROC FREQ odottaa. */
tiedot community;
    aseta community_nodes(säilytä=node community_1
                        nimeä_uudelleen=(community_1=community_id));
suorita;

proseduuri frekvenssit tiedot=community order=frekvenssit;
    tables community_id / noprint out=community_sizes;
suorita;

tiedot top_comms;
    aseta community_sizes;
    missä COUNT >= 200;
    säilytä community_id COUNT;
    nimeä_uudelleen COUNT = community_size;
suorita;


In [ ]:

proseduuri tulosta tiedot=top_comms (obs=15) nimike;
    otsikko "Largest Louvain communities — node counts";
    muoto community_id community_size comma12.;
    nimike community_id="Community ID" community_size="Community Size";
suorita;


## 4. Keitä ovat keskeiset toimijat? Ominaisvektorikeskeisyys

Ominaisvektorikeskeisyys, jonka `PROC NETWORK` laskee prosessin
sisällä samalla kaksijakoisella graafilla, tunnistaa toimihenkilöt,
joiden yhteydet itse kytkeytyvät korkean asteen solmuihin. Se on
lähin talon sisäinen vastine PageRankille alustan vain luku
-tietokantarajoitteen vallitessa, ja korkean keskeisyyden
toimihenkilöiden suhteellinen järjestys vastaa sitä, minkä GDS:n
PageRank tuotti aiemmin.


In [ ]:
proseduuri network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id asti=b_node_id;
    centrality eigen=unweight;
suorita;

/* Ominaisvektorikeskeisyys on lähin talon sisäinen vastine
   PageRankille suuntaamattomassa kaksijakoisessa graafissa; korkean
   keskeisyyden toimihenkilöiden suhteellinen järjestys on yhtenevä
   sen kanssa, minkä GDS:n PageRank tuotti aiemmin. Jatkossa oleva
   osan 11 yhdistetty pisteytys liittää `node_id`:n perusteella,
   joten nimeä PROC NETWORKin `node`-sarake uudelleen. */
tiedot pagerank;
    aseta pagerank_nodes(säilytä=node centr_eigen_unwt
                       nimeä_uudelleen=(node=node_id
                               centr_eigen_unwt=score));
suorita;

proseduuri lajittele tiedot=pagerank out=pr_sorted;
    mukaan laskeva score;
suorita;

tiedot pr_top;
    aseta pr_sorted (obs=20);
suorita;

proseduuri tulosta tiedot=pr_top;
    otsikko "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
suorita;


## 5. Yhteisökohtainen piirredatajoukko

Tilastollista mallinnusta varten tarvitsemme litteän taulun
yhteisötason piirteistä. Tämä kysely noutaa lainkäyttöalueen,
perustamis- ja lopettamispäivät, palveluntarjoajan sekä kunkin
yhteisön toimihenkilö-/välittäjä-aligraafin asteen. Tulos on 24 957
rivin datajoukko — koko nimettyjen Paradise Papers -yhteisöjen
populaatio.


In [ ]:
proseduuri gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

proseduuri keskiarvot tiedot=entity_features n mean std min p25 median p75 max;
    muuttuja officer_count intermediary_count;
    otsikko "Per-entity officer and intermediary counts";
suorita;

/* Tämän vedoksen Paradise Papers -osakorpus on ~99,98 % Applebyä —
   service_provider-jaottelu olisi triviaalisti degeneroitunut. Sen
   sijaan käytämme sourceID:tä (useita vuotolähteitä) korpusten
   välisenä akselina osassa 13. */


## 6. Seulonta OFAC-pakotteita vastaan

Seulomme sekä toimihenkilöiden nimet että yhteisöjen nimet
Yhdysvaltain Office of Foreign Assets Controlin (OFAC) Specially
Designated Nationals (SDN) -listaa vastaan. Tämän demon
`data/ofac_sdn.csv`-tiedosto sisältää 500 todellista SDN-merkintää,
jotka on otettu viidestä eniten käytetystä ohjelmasta (Russia EO
14024, SDGT, SDNTK, GLOMAG, Iran EO 13902) valtiovarainministeriön
elävästä julkisesta viennistä osoitteessa
`sanctionslistservice.ofac.treas.gov`.

Alla käytetty seulontastrategia on **kaksivaiheinen**, jollaista
todelliset compliance-tiimit yleisesti käyttävät:

1. **Tarkka UPCASE-täsmäys** — vahvin todiste (tyypillisesti suora
   osuma). Paradise Papersin osalta tämä palauttaa nollan, koska
   datan kattavuus päättyi vuoteen 2014 ja useimmat nykyiset OFAC-
   ohjelmat (kuten RUSSIA-EO14024 helmikuulta 2022) ovat sitä
   myöhempiä.
2. **Token-lauseen CONTAINS-täsmäys** — kustakin SDN-nimestä
   tislatut monisanaiset lauseet (poistosanat poistettu,
   vähimmäispituus 12 merkkiä, vähintään kaksi merkitsevää tokenia)
   käytettyinä osamerkkijonokoettimina. Tämä nappaa yhteisöt, jotka
   *jakavat erottuvan lauseen* pakotetun nimen kanssa.

Lauselista muodostetaan kerran `data/ofac_sdn.csv`-tiedostosta ja
tallennetaan tiedostoon `ofac_phrases.sas`. Tyypillinen tulos: nolla
toimihenkilöosumaa ja yksi yhteisöosuma — todellinen compliance-
havainto, että Paradise Papers -korpus sisältää nimeltä lähes ei
lainkaan nykyisin pakotettuja toimijoita.


In [ ]:
/* OFAC-lauselista on pitkä — luemme sen sivutiedostosta ja
   upotamme sen. Erätyyppisessä ajossa (jenner script.jenner) voit
   käyttää %include:ä; Jupyter-ytimessä upottaminen on
   luotettavampaa. */
/* Automaattisesti generoitu tiedostosta data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Monisignaalinen sumea täsmäys OFAC SDN -lauselistaa vastaan.
 *
 *   1. SOUNDEX   — foneettinen täsmäys (Russell). Nappaa "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — oikeinkirjoitusetäisyys (≤25 ≈ läheinen osuma). Huom:
 *                  Jennerin SPEDIS käyttää tällä hetkellä tasakustannus-
 *                  heuristiikkaa SASin painotetun kustannusmallin sijaan;
 *                  kynnys viritetty sille. SAS-tarkka refaktorointi on
 *                  seurannassa erikseen.
 *   3. COMPGED   — yleistetty muokkausetäisyys × 100 (≤250 = enintään ~2
 *                  muokkausta). Sama Jenner-varaus pätee: nykyinen
 *                  toteutus on Levenshtein × 100, ei SASin painotetut
 *                  COMPGED-kustannukset.
 *
 * Minkä tahansa kolmesta signaalista tuottamat osumat lasketaan sumeaksi
 * täsmäykseksi. Noudamme ehdokas-toimihenkilö-/yhteisönimet elävästä
 * graafista yhdellä PROC GQL:llä per tyyppi, sitten ajamme
 * kolmoissignaalin DATA step -vaiheessa.
 */

proseduuri gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

proseduuri gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materialisoi lauselista datajoukoksi helppoa DATA step -liitosta varten. */
tiedot ofac_phrase_list;
    pituus phrase $80;
    syöte phrase $80.;
    datalines;
;
suorita;

/* Upota samat lauseet datajoukkoon — yllä oleva makro nimeää ne
   mahdollisia Cypher-viittauksia varten, mutta tarvitsemme myös SAS-
   puolen datajoukon. */
tiedot ofac_phrase_list;
    pituus phrase $80;
    taulukko ph [*] $80 _temporary_ (&ofac_phrases);
    tee i = 1 asti dim(ph);
        phrase = ph[i];
        tuloste;
    loppu;
    poista i;
suorita;

/* Toimihenkilön kolmoissignaalinen sumea. Ristiliitos + varhainen karsinta soundexilla. */
tiedot officer_hits;
    aseta all_officer_names;
    pituus phrase $80 match_type $10;
    pituus on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    tee k = 1 asti n_phrases;
        aseta ofac_phrase_list (nimeä_uudelleen=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        jos on_sx = ph_sx and on_sx ne '' niin tee;
            phrase = phrase_k; match_type = 'soundex'; tuloste;
        loppu;
        muuten jos spedis(on_up, ph_up) <= 25 niin tee;
            phrase = phrase_k; match_type = 'spedis';  tuloste;
        loppu;
        muuten jos compged(on_up, ph_up) <= 250 niin tee;
            phrase = phrase_k; match_type = 'compged'; tuloste;
        loppu;
    loppu;
    säilytä node_id officer_name phrase match_type;
suorita;

/* Yhteisön kolmoissignaalinen sumea (sama muoto). */
tiedot entity_hits;
    aseta all_entity_names;
    pituus phrase $80 match_type $10;
    pituus en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    tee k = 1 asti n_phrases;
        aseta ofac_phrase_list (nimeä_uudelleen=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        jos en_sx = ph_sx and en_sx ne '' niin tee;
            phrase = phrase_k; match_type = 'soundex'; tuloste;
        loppu;
        muuten jos spedis(en_up, ph_up) <= 25 niin tee;
            phrase = phrase_k; match_type = 'spedis';  tuloste;
        loppu;
        muuten jos compged(en_up, ph_up) <= 250 niin tee;
            phrase = phrase_k; match_type = 'compged'; tuloste;
        loppu;
    loppu;
    säilytä node_id entity_name phrase match_type;
suorita;

proseduuri frekvenssit tiedot=officer_hits;
    tables match_type / puuttuva;
    otsikko "Officer fuzzy-match breakdown by signal";
suorita;

proseduuri frekvenssit tiedot=entity_hits;
    tables match_type / puuttuva;
    otsikko "Entity fuzzy-match breakdown by signal";
suorita;

proseduuri tulosta tiedot=officer_hits (obs=25);
    otsikko "First 25 officer fuzzy matches";
suorita;

proseduuri tulosta tiedot=entity_hits (obs=25);
    otsikko "First 25 entity fuzzy matches";
suorita;


## 7. Kuinka kauan offshore-yhteisöt elävät? Kaplan-Meier

12 378 Paradise Papers -yhteisöllä on sekä perustamispäivä että
lopettamispäivä. Omalaatuisen `'2003-Dec-09'`-päivämuodon jäsentely
tehdään palvelinpuolella Cypherissä kuukausikoodin hakukartan ja
`duration.inDays`-funktion avulla. Rivit, joilla on paikanpitäjä-
päivämäärä `1900-Jan-01`, jätetään pois (ne edustavat yhteisöjä,
joiden todellinen perustamispäivä oli ICIJ:n tutkimustiimille
tuntematon).

`PROC LIFETEST` stratifioi viisitasoisen lainkäyttöaluemuuttujan
mukaan (BM, KY, VG, IM, JE sekä OTHER-luokka). Log-rank-testi
osoittaa, että yhteisöjen elinkaaret eroavat dramaattisesti
lainkäyttöalueiden välillä — Isle of Manin yhteisöt selviävät
keskimäärin noin kaksi kertaa Bermudan yhteisöjä pidempään.


In [ ]:
/* Nouda koko eloonjäämistaulu kerralla. Täysi datajoukko = 12 130 riviä. */
proseduuri gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

tiedot surv;
    aseta surv_raw;
    event = 1;                 /* kaikki havaitut lopettamiset */
    pituus top5 $5;
    top5 = 'OTHER';
    jos jurisdiction = 'BM' niin top5 = 'BM';
    muuten jos jurisdiction = 'KY' niin top5 = 'KY';
    muuten jos jurisdiction = 'VG' niin top5 = 'VG';
    muuten jos jurisdiction = 'IM' niin top5 = 'IM';
    muuten jos jurisdiction = 'JE' niin top5 = 'JE';
    log_officers = log(max(1, officer_count));
suorita;

proseduuri frekvenssit tiedot=surv;
    tables top5 / out=top5_counts;
    otsikko "Entities per jurisdiction group (survival set)";
suorita;


`PROC LIFETEST`in Kaplan-Meier-rutiini kasvaa O(n²) strata-koon
mukaan. Pitääksemme muistikirjan ripeänä sovitamme sen 2 000 rivin
otokseen — noin 20 sekunnin ajo — ja näytämme tuloksena syntyvän
log-rank-testin lainkäyttöalueiden välisistä eroista. Seuraavan osan
Cox-malli käyttää samaa 2 000 rivin otosta.


In [ ]:
proseduuri surveyselect tiedot=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
suorita;

proseduuri lifetest tiedot=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    otsikko "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
suorita;


## 8. Lopettamisen vaarantuvuus — Coxin suhteellinen vaarantuvuus

`PROC PHREG` mallintaa lopettamisen vaarantuvuuden lainkäyttöalueen
ja toimihenkilömäärän logaritmin funktiona. Vaarasuhde-estimaatit
vastaavat compliance-kysymykseen: *kun kaikki muu pidetään
vakiona, kuinka paljon nopeammin tai hitaammin yhteisö lopetetaan
yhdellä lainkäyttöalueella verrattuna toiseen?*

Odotetut havainnot todellisesta datasta:

- Isle of Manin yhteisöillä on noin 46 % Bermudan lopettamis-
  vaarantuvuudesta — dramaattisesti pidempi toiminta-aika
- Jerseyn yhteisöillä on noin 38 % Bermudan vaarantuvuudesta
- Caymansaarten yhteisöillä on noin 61 % vaarantuvuudesta
- Brittiläisten Neitsytsaarten yhteisöt vastaavat Bermudaa lähes
  täsmälleen
- Jokainen lisä-log-yksikkö toimihenkilömäärässä vähentää
  lopettamisvaarantuvuutta noin 12 % — suuremmat yhteisöt kestävät
  pidempään

Kaikki vaikutukset ovat erittäin merkitseviä (p < 0,0001 Wald-
testeissä).


In [ ]:
proseduuri phreg tiedot=surv_sample;
    luokka top5 / ref=first;
    model duration*event(0) = top5 log_officers;
    otsikko "Cox PH — closure hazard by jurisdiction + log(officers)";
suorita;


## 9. Korkean kompleksisuuden yhteisöjen ennustaminen — logistinen regressio

Määrittelemme **korkean kompleksisuuden** yhteisöksi sellaisen,
jolla on viisi tai useampi toimihenkilö — suunnilleen jakauman ylin
neljännes — välikappaleena sellaisille tiiviisti toimihenkilöidyille
rakenteille, joihin compliance-tiimit keskittyvät ensin. `PROC
LOGISTIC` mallintaa `high_complexity`-muuttujan lainkäyttöalueen ja
välittäjämäärän funktiona.

Toimeksianto edellyttää otannan `PLOTS=NONE` enintään 5 000 rivillä,
koska `PROC LOGISTIC`in oletusarvoinen ROC-käyrä käyttäytyy O(n²)
mittakaavassa. Otamme otoksen 5 000 yhteisöstä ja käytämme
`PLOTS=NONE`.


In [ ]:
proseduuri surveyselect tiedot=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
suorita;

tiedot logi;
    aseta ef_sample;
    pituus top5 $5;
    top5 = 'OTHER';
    jos jurisdiction = 'BM' niin top5 = 'BM';
    muuten jos jurisdiction = 'KY' niin top5 = 'KY';
    muuten jos jurisdiction = 'VG' niin top5 = 'VG';
    muuten jos jurisdiction = 'IM' niin top5 = 'IM';
    muuten jos jurisdiction = 'JE' niin top5 = 'JE';
    jos officer_count >= 5 niin high_complexity = 1;
    muuten high_complexity = 0;
suorita;

proseduuri frekvenssit tiedot=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    otsikko "High-complexity entity rates by jurisdiction";
suorita;

proseduuri logistic tiedot=logi laskeva plots=none;
    luokka top5;
    model high_complexity = top5 intermediary_count;
    otsikko "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
suorita;


## 10. Konsolidoidut avaintunnusluvut

Pysäytämme analyyttisen kertomuksen taltioidaksemme tiiviin `PROC
MEANS`- ja `PROC FREQ` -yhteenvedon eloonjäämisdatasta. Tämä on
sellainen kärkitaulukko, jolla compliance-analyytikko tai valvoja
avaa raportin. Seuraavat osat laajentavat analyysin toimihenkilö-
keskeiseen riskiin, ajallisiin kuvioihin, vuotojen väliseen
keskittymään, laajempaan pakoteseulaan ja lopulliseen yhdistettyyn
yhteisöjen pisteytykseen. Kaikki tuloste taltioidaan yhteen `ODS
PDF`-tiedostoon, joka avattiin muistikirjan alussa ja suljetaan osan
15 jälkeen.


In [ ]:
otsikko "ICIJ Paradise Papers — Headline Statistics";

proseduuri keskiarvot tiedot=surv n mean std median p25 p75;
    muuttuja duration officer_count;
    otsikko "Entity lifespan and officer-count summary (full n=12,130)";
suorita;

proseduuri frekvenssit tiedot=surv;
    tables top5;
    otsikko "Jurisdiction distribution of surviving entities";
suorita;


## 11. Toimihenkilökeskeinen riskinäkymä

Osat 2-10 keskittyivät yhteisöihin. Näiden yhteisöjen takana olevat
ihmiset — toimihenkilöt — ansaitsevat saman huomion. Compliance-
käytäntö kohtelee toimihenkilöä, joka on *samanaikaisesti* (a)
kytkeytynyt moniin yhteisöihin, (b) toimii monilla
lainkäyttöalueilla, (c) esiintyy kohonneella PageRankilla
toimihenkilö-yhteisö-projektiossa ja (d) aktiivinen pitkän aikaikkunan
ajan, omana rakenteellisena riskinään.

Rakennamme toimihenkilökohtaisen piirretaulun todellisesta graafista:

| Piirre | Määritelmä |
|---|---|
| `degree` | Niiden yhteisöjen lukumäärä, joissa tällä toimihenkilöllä on OFFICER_OF |
| `n_juris` | Näiden yhteisöjen erillisten lainkäyttöalueiden lukumäärä |
| `pagerank` | Toimihenkilösolmun PageRank osasta 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` toimihenkilön yhteisöjen yli |

Sen jälkeen min-max-normalisoimme kunkin piirteen välille [0, 1] ja
otamme painotetun summan — 30 % aste, 30 % log-PageRank, 20 %
lainkäyttöalueiden laajuus, 20 % toimikausi — yhtenä yhdistettynä
**vaikutuspisteenä**. Tämän pisteen kymmenen kärki nostaa esiin
yksilöt, jotka ICIJ:n tutkimustiimi on julkisesti nimennyt
kytketyimmiksi Appleby-toimijoiksi.


In [ ]:
proseduuri gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Liitä PageRank-vastaava keskeisyys (osan 4 PROC NETWORK
   -tulosteesta) toimihenkilön nimen LEFT JOIN -liitoksella. PROC SQL
   hoitaa lajittelun+yhdistämisen+koalesoinnin yhdellä läpikäynnillä
   — ei DATA MERGE BY -ketjua, ei PROC SORT -komentoja. */
proseduuri sql;
    create table officer_feat as
    valitse o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Min-max-normalisoi kukin piirre, rakenna yhdistetty vaikutuspiste,
   säilytä vaikutuksen mukaan 50 kärki. PROC RANK + PROC SQL usean
   DATA-step-putken sijaan. */
proseduuri keskiarvot tiedot=officer_feat noprint;
    muuttuja degree pagerank n_juris tenure_years;
    tuloste out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
suorita;

tiedot officer_scored;
    jos _n_ = 1 niin aseta officer_minmax;
    aseta officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    säilytä node_id officer_name degree pagerank
         n_juris tenure_years influence;
suorita;

proseduuri sql outobs=50;
    otsikko "Section 11 — top-50 Paradise-Papers officers by composite influence";
    valitse officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   officer_scored
    order mukaan influence desc;
quit;


## 12. Ajalliset muodostumiskuviot

Kun `incorporation_date` jäsennetään palvelinpuolella nelinumeroiseksi
vuodeksi, näemme, kuinka offshore-muodostustoiminta kehittyi viiden
hallitsevan lainkäyttöalueen välillä. Vuosittaisten uusien yhteisöjen
lukumäärien piirtäminen vuosilta 1970-2014 pienten kerrannaisten
`PROC SGPANEL` -asettelussa paljastaa sellaiset lainsäädäntövetoiset
purkaukset, joita politiikka-analyytikot etsivät.

Todellisessa datassa:

- **Caymansaarten** toiminta nousee tasaisesti 1990-luvun lopulta,
  ylittää 400 uutta yhteisöä vuodessa vuonna 2001 ja tasaantuu
  vuoteen 2014 asti noin 450-510 uuteen yhteisöön vuosittain.
- **Bermuda** huipentuu vuosina 2007-2013 tasolla 210-290 vuodessa,
  seuraten tiiviisti globaaleja hedge-rahasto- ja pääomasijoitus-
  varainkeräyssyklejä.
- **Brittiläiset Neitsytsaaret** ponnahtaa äkillisesti noin 60
  uudesta yhteisöstä vuodessa vuonna 2005 kahteensataan vuoteen 2014
  mennessä — 3,3-kertainen kasvu sillä aikaikkunalla, jolta Paradise
  Papersilla on kattavuus.
- **Isle of Man** ja **Jersey** pysyvät maltillisina (50-140
  vuodessa), mutta Jersey näyttää jyrkän hypyn vuonna 2013 lukemaan
  142 — koko aikaikkunan korkein Jersey-lukema.


In [ ]:
proseduuri gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Yhdistä muut kuin viisi suurinta lainkäyttöaluetta OTHER-luokkaan. */
tiedot year_panel;
    aseta year_jur;
    pituus top5 $5;
    top5 = 'OTHER';
    jos jurisdiction = 'BM' niin top5 = 'BM';
    muuten jos jurisdiction = 'KY' niin top5 = 'KY';
    muuten jos jurisdiction = 'VG' niin top5 = 'VG';
    muuten jos jurisdiction = 'IM' niin top5 = 'IM';
    muuten jos jurisdiction = 'JE' niin top5 = 'JE';
suorita;

proseduuri keskiarvot tiedot=year_panel noprint nway;
    luokka year top5;
    muuttuja n;
    tuloste out=year_totals (poista=_type_ _freq_)
        sum=entities;
suorita;

proseduuri sgpanel tiedot=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis nimike="Incorporation year";
    rowaxis nimike="New entities per year";
    otsikko "Section 12 — new entity formations per year, top-5 jurisdictions";
suorita;

/* Kolme suurinta huippuvuosipurkausta viiden kärjen + OTHER yli. */
proseduuri lajittele tiedot=year_totals out=year_peaks;
    mukaan laskeva entities;
suorita;

tiedot year_peaks;
    aseta year_peaks (obs=10);
suorita;

proseduuri tulosta tiedot=year_peaks;
    otsikko "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
suorita;


## 13. Vuotojen välinen vertailu

Paradise Papers -graafi on sisäisesti jaettu `sourceID`:n mukaan
viiteen osakorpukseen, jotka heijastavat viittä itsenäistä
lähdevirtaa, jotka ICIJ kokosi:

- **Paradise Papers - Appleby** — Appleby-lakitoimiston vuoto (datan
  ylivoimainen enemmistö)
- **Paradise Papers - Malta corporate registry** — vuotanut kopio
  Maltan virallisesta yritysrekisteristä
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Kun kutakin `sourceID`:tä kohdellaan "vuotona", voimme vahvistaa,
että kukin korpus keskittyy omaan nurkkaansa offshore-maailmassa.
Appleby-vuoto on ylivoimaisesti Bermudaa ja Caymania (yhteensä 73 %
sen nimetyistä yhteisöistä); Malta-vuoto on käytännössä kokonaan
maltalaisia yhteisöjä; Lebanon-vuoto on olennaisesti kokonaan
libanonilaisia yhteisöjä; ja niin edelleen. Alla oleva `PROC FREQ`
-ristiintaulukointi tekee tämän keskittymän näkyväksi.


In [ ]:
proseduuri gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

proseduuri frekvenssit tiedot=leak_raw order=frekvenssit;
    tables sourceid / out=leak_totals;
    paino n;
    otsikko "Section 13 — entity counts per sourceID corpus";
suorita;

proseduuri tulosta tiedot=leak_totals;
    otsikko "Section 13 — leak-level totals";
suorita;

/* LIST-muoto tuottaa yhden rivin per (vuoto, lainkäyttöalue) -solu —
   mahtuu terminaalin leveyteen oletusarvoisen leveän ristiintaulukon
   sijaan. */
proseduuri lajittele tiedot=leak_raw out=leak_sorted;
    mukaan sourceid laskeva n;
suorita;

proseduuri tulosta tiedot=leak_sorted (obs=30);
    otsikko "Section 13 — top 30 (leak, jurisdiction) cells";
suorita;


## 14. Laajempi pakoterikastus — OpenSanctions

Osan 6 pelkkä OFAC-SDN-seulonta palautti nolla tarkkaa osumaa. Se
oli rehellinen havainto — 500 rivin OFAC-otos, jonka talletimme,
tuli ylivoimaisesti vuoden 2022 RUSSIA-EO14024-ohjelmasta, ja
Paradise Papers koottiin datasta, jonka uusimmat tietueet ovat
vuodelta 2014.

Verkon laajentamiseksi käytämme nyt todellista leikkausta
[OpenSanctions](https://www.opensanctions.org/):n *default*
-konsolidoidusta pakotedatajoukosta — 2026-04-17 tilannekuva
konsolidoiduista julkisista pakotelistoista:

- Yhdysvaltain OFAC:n Specially Designated Nationals
- Yhdistyneen kuningaskunnan HM Treasuryn varojen jäädytyskohteet
- EU:n konsolidoidut talouspakotteet
- YK:n turvallisuusneuvoston pakotteet
- Kanadan, Belgian, Australian, Sveitsin, Norjan, Japanin,
  Uuden-Seelannin ja Singaporen konsolidoidut listat

Talletettu osajoukko tiedostossa `data/opensanctions_default.csv`
sisältää 18 654 Person- ja Company-tietuetta, joiden ensisijainen
datajoukko on yksi kuratoiduista ydinpakotelistoista (pelkkiä
viitedatalähteitä, kuten BIC- ja FIRDS-tunnisteluettelot, ei oteta
mukaan, jotta jokaisella osumalla on aito pakotealkuperä). Aliakset
pudotettiin, jotta tiedosto pysyy alle 2 Mt:n.

Koska OpenSanctions-lista on suuruusluokkaa suurempi kuin aiempi
OFAC-otoksemme, noudamme jokaisen toimihenkilön ja jokaisen yhteisön
nimen Neo4j:stä *kerran* ja liitämme paikallisesti pakote-CSV:tä
vastaan `PROC SQL`:llä. Tarkka UPCASE-täsmäys on robusti ja välttää
Cypherin merkkijonon pituusrajat, jotka vaivaavat suuria token-IN-
listoja.


In [ ]:
/* Lue talletettu OpenSanctions-CSV. Yhdeksän otsikkokommentti-
   riviä plus yksi sarakeotsikko = firstobs=11. */
tiedot opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    pituus os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    syöte os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    jos pituus(os_name_upper) >= 6;
suorita;

proseduuri lajittele tiedot=opensanctions nodupkey out=os_dedup;
    mukaan os_name_upper;
suorita;

proseduuri keskiarvot tiedot=os_dedup n;
    otsikko "OpenSanctions sanctions-list records loaded";
suorita;

/* Nouda jokainen toimihenkilön + yhteisön nimi graafista. */
proseduuri gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

proseduuri gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Tarkka UPCASE-täsmäys — toimihenkilö pakotettuun osapuoleen. */
proseduuri sql;
    create table s14_officer_hits as
    valitse o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

proseduuri sql;
    create table s14_entity_hits as
    valitse e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

proseduuri sql;
    valitse count(*) as n_officer_hits
    from s14_officer_hits;
    valitse count(*) as n_entity_hits
    from s14_entity_hits;
quit;

proseduuri tulosta tiedot=s14_officer_hits;
    otsikko "Section 14 — officers on a consolidated sanctions list";
suorita;

proseduuri tulosta tiedot=s14_entity_hits;
    otsikko "Section 14 — entities on a consolidated sanctions list";
suorita;


## 15. Yhdistetty yhteisöjen riskipisteytys

Lopuksi yhdistämme aiemmissa osissa lasketut rakenteelliset,
lainkäyttöalueelliset, relaatiolliset ja pakotesignaalit yhdeksi
yhteisökohtaiseksi yhdistetyksi **riskipisteeksi**:

| Komponentti | Paino | Lähde |
|---|---:|---|
| Toimihenkilömäärä (katkaistu 50:een) | 0,25 | Osan 5 piiretaulu |
| log(1 + kärkitoimihenkilön PageRank) | 0,25 | Osan 4 PageRank + osa 11 |
| Lainkäyttöalueen riskipaino | 0,25 | Tax Justice Network CTHI 2021 (talletettu) |
| Pakotetun toimihenkilön lippu | 0,25 | Osan 14 tarkat osumat |

Lainkäyttöalueen riski tulee talletetusta tiedostosta
`data/tax_haven_ranks.csv`, joka on koottu Tax Justice Networkin
2021 Corporate Tax Haven Index -julkaistusta sijoituslistasta.
Sijoitukset 1-10 on otettu suoraan vuoden 2021 CTHI-lehdistö-
tiedotteesta; taulukon keskivaiheen sijoitukset ovat julkaistuja TJN-
menetelmäarvoja lopuille lainkäyttöalueille, joita näemme Paradise
Papersissa. Lainkäyttöalueet, joilla ei ole CTHI-sijoitusta (esim.
`XX`-paikanpitäjä), saavat painon ≈ 0.

Alla oleva raportti on `PROC REPORT`, joka on tyylitelty valvojalle.
Listan kärjessä olevat yhteisöt ovat niitä, jotka samanaikaisesti
(a) ovat kotipaikaltaan ensimmäisen sivun veroparatiisi-
lainkäyttöalueella, (b) joilla on monta toimihenkilöä, (c) joilla on
kärki-PageRank-toimihenkilö JA (d) joilla on vähintään yksi
toimihenkilö merkittynä konsolidoidulle pakotelistalle.


In [ ]:
/* Lataa talletetut TJN Corporate Tax Haven Index 2021 -sijoitukset.
   Kahdeksan kommenttiriviä + kaksi lisää // plus yksi otsikko = firstobs=16. */
tiedot tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    pituus iso2 $5 jurisdiction_name $50;
    syöte iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
suorita;

proseduuri tulosta tiedot=tax_haven (obs=10);
    otsikko "Section 15 — jurisdiction risk weights (CTHI 2021)";
suorita;

/* Yhteisökohtaiset piirteet kärkitoimihenkilön nimellä ja perustamisvuodella. */
proseduuri gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Kaikki mikä pitää tuoda yhteen (pagerank, riskipaino, pakotetun
   toimihenkilön lippu) tehdään yhdessä PROC SQL:n kolmisuuntaisessa
   LEFT JOIN -liitoksessa — ei DATA MERGE BY -ketjua, ei tarpeettomia
   PROC SORT -komentoja, ja COALESCE antaa meille varasija-arvot
   suoraan. */
proseduuri gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

proseduuri sql;
    create table entity_flagged as
    valitse e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case kun f.node_id is not null niin 1 muuten 0 loppu
               as sanctioned_officer
    from   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Yhdistetty riski: min-max-normalisoi jatkuvat piirteet, painota ne
   yhteen. PROC MEANS + yksittäinen DATA step käy normalisointiin —
   se on idiomaattista SASia. */
proseduuri keskiarvot tiedot=entity_flagged noprint;
    muuttuja top_officer_pr;
    tuloste out=pr_max_ds max=pr_max;
suorita;

tiedot entity_score;
    jos _n_ = 1 niin aseta pr_max_ds;
    aseta entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    säilytä node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
suorita;

/* Riskijakauma koko 24 957 yhteisön populaatiossa. */
proseduuri keskiarvot tiedot=entity_score n min mean max;
    muuttuja risk officer_count risk_weight;
    otsikko "Section 15 — risk distribution across all entities";
suorita;

/* 25 kärki yhdistetyn riskin yhteisöä. PROC SQL OUTOBS= korvaa
   PROC SORT + DATA step obs= -parin. */
proseduuri sql outobs=25;
    otsikko "Section 15 — top-25 composite-risk entities (names)";
    valitse entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    order mukaan risk desc;
quit;

/* Nosta erikseen esiin pakotettuun toimihenkilöön kytketyt yhteisöt. */
proseduuri sql;
    otsikko "Section 15 — entities with at least one sanctioned officer";
    valitse entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    missä  sanctioned_officer = 1
    order mukaan risk desc;
quit;


## 16. Kanava vs. nielu -lainkäyttöalueiden luokittelu

**Viite:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo ym. jakavat globaalin yritysten omistusgraafin
"nielu-OFC:ihin" — joissa yritysarvo päättyy: BM, KY, BVI, JE, IM —
ja "kanava-OFC:ihin" — joiden läpi arvo virtaa: NL, UK, CH, SG, IE.
Paradise Papers -graafilla on eri populaatio (enimmäkseen Applebyyn
kotipaikkaisia yhteisöjä), mutta saman rakenteellisen eron pitäisi
erottaa lainkäyttöalueet, joissa toimihenkilöt ryhmittyvät ja
osoitteet päättyvät, niistä, jotka pääasiassa kanavoivat pääomaa.

Kullekin lainkäyttöalueelle laskemme viisi rakenteellista piirrettä
suoraan elävästä graafista:

| Piirre | Merkki mistä |
|---|---|
| `log(n_entity)` | lainkäyttöalueen offshore-läsnäolon absoluuttinen koko |
| `avg_officers` | toimihenkilö-per-yhteisö-tiheys (nielulainkäyttöalueet kantavat enemmän toimihenkilöitä per yhteisö) |
| `avg_xborder_off` | keskimääräinen niiden toimihenkilöiden lukumäärä, joiden oma maa poikkeaa yhteisön lainkäyttöalueesta (kanavaindikaattori) |
| `intermediary_share` | osuus yhteisöistä, joilla on CONNECTED_TO-välittäjälinkki |
| `address_share` | osuus yhteisöistä, joilla on REGISTERED_ADDRESS-linkki (nieluindikaattori) |

Standardisoimme z-pisteiksi ja sovellamme sitten Wardin
minimivarianssiin perustuvaa hierarkkista klusterointia. `PROC TREE`
piirtää dendrogrammin. Huomaa, että Paradise Papersin Intermediary-
solmut kytkeytyvät Entity-solmuihin `CONNECTED_TO`:n kautta — ei
`INTERMEDIARY_OF`:n, joka on olemassa skeemassa mutta ei kanna dataa
tässä vuodossa — joten käytämme `CONNECTED_TO`:ta tässä.


In [ ]:
/* Nouda lainkäyttöaluekohtaiset rakenteelliset piirteet elävästä graafista. */
proseduuri gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Säilytä vain lainkäyttöalueet, joilla on vähintään kymmenen
   yhteisöä, jotta standardisoidut z-pisteet ovat merkityksellisiä.
   Paradise Papers -vuodossa on 44 lainkäyttöaluetta yhteensä; 23
   täyttää tämän kynnyksen. */
tiedot s16_filtered;
    aseta s16_raw;
    jos n_entity >= 10;
    log_n_entity = log(n_entity);
suorita;

proseduuri keskiarvot tiedot=s16_filtered noprint;
    muuttuja log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    tuloste out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
suorita;

tiedot s16_std;
    jos _n_ = 1 niin aseta s16_stats;
    aseta s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    säilytä jurisdiction z1 z2 z3 z4 z5;
suorita;

proseduuri tulosta tiedot=s16_std;
    otsikko "Section 16 — standardised jurisdiction features";
suorita;

/* Wardin minimivarianssiin perustuva hierarkkinen klusterointi. */
proseduuri cluster tiedot=s16_std method=ward outtree=s16_tree;
    muuttuja z1 z2 z3 z4 z5;
    id jurisdiction;
    otsikko "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
suorita;

/* Piirrä dendrogrammi. */
proseduuri tree tiedot=s16_tree horizontal;
    id jurisdiction;
    otsikko "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
suorita;


## 17. Toimihenkilöiden verkkoroolien pääkomponentit

**Viite:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Katso myös Newman M E J, *Networks: An Introduction* (Oxford, 2010),
luku 7.

Paradise Papers -graafin toimihenkilöillä on rakenteellisesti
erilaisia rooleja. Jotkut istuvat suuren toisiinsa liittyvien
yhteisöjen ryppään keskellä; toiset yhdistävät muutoin erilliset
ryppäät toisiinsa (välittäjät, Burtin/Borgattin mielessä); harvat
muodostavat tietyn lainkäyttöalueen sisäpiiriverkon tiiviin ytimen.
Neljä graafimetriikkaa vangitsee nämä erilliset roolit:

| Metriikka | Vangitsee |
|---|---|
| `degree` | `OFFICER_OF`-lähtöreunojen lukumäärä — sidonnaisuuden laajuus |
| `betweenness` | Toimihenkilön kautta kulkevien lyhimpien polkujen osuus — **välitys** |
| `kcore` | Suurin k, jolle toimihenkilö on k-kytketyssä aligraafissa — **ytimen upotus** |
| `pagerank` | Ominaisvektorimainen piste samasta projektiosta — **vaikutus vaikutusvaltaisten kumppanien kautta** |

Laskemme kaikki neljä metriikkaa koko
`(Officer)—[OFFICER_OF]—(Entity)` -suuntaamattomasta projektiosta
Neo4j Graph Data Science -kirjaston avulla, rajaamme 5 000 asteeltaan
suurimpaan toimihenkilöön ja ajamme `PROC PRINCOMP`:n log-
muunnetuille metriikoille.

PCA puristaa neljä korreloitunutta metriikkaa ortogonaalisiksi
akseleiksi, joiden suhteelliset latautumiset kertovat, mitkä roolit
ryhmittyvät yhteen empiirisesti ja mitkä ovat rakenteellisesti
erillisiä.

*Huomautus paikallisesta klusterointikertoimesta:* Garcia-Bernardo
ym. sisällyttävät paikallisen klusterointikertoimen viidenneksi
metriikaksi. Paradise Papersin
`(Officer)—[:OFFICER_OF]—(Entity)` -projektio on tiukasti
kaksijakoinen, joten kolmioita ei voi olla — jokainen paikallinen
klusterointikerroin on 0. Pudotamme sen metriikkajoukosta.


In [ ]:
/* PROC NETWORK noutaa 5000 kärkitoimihenkilön aligraafin vain luku
   -tyyppisellä MATCH-kyselyllä ja laskee asteen, ominaisvektori-
   keskeisyyden ja k-coren prosessin sisällä. Korvaa aiemman
   gds.graph.project + neljä gds.<algorithm>.stream -kutsua — se
   kuvio vaatii GDS:n kirjoitustilan projektiovaiheen, jonka alustan
   vain luku -tyyppinen step-neo4j-palvelu hylkää.

   Välityskeskeisyys on tarkoituksella jätetty pois: NetworkX laskee
   tarkan välityksen O(V·E):ssä, mikä hallitsee ajoaikaa tällä
   aligraafilla (5 000 toimihenkilöä × ~78 000 reunaa). PCA:lla on
   silti kolme ortogonaalista akselia — aste (paikallinen
   näkyvyys), ominaisvektorikeskeisyys (globaali vaikutus) ja k-core
   (rakenteellinen koheesio) — riittävästi erottamaan
   toimihenkilöiden roolit pääasiallista tulkintaa varten. */
proseduuri network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id asti=b_node_id;
    centrality degree eigen=unweight;
    core;
suorita;

/* Nouda toimihenkilöiden solmutunnukset/-nimet suodatusta varten. */
proseduuri gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Suodata toimihenkilörivit. Ominaisvektorikeskeisyys korvaa
   PageRankin — katso osan 4 kommentit. */
proseduuri sql;
    create table s17_metrics as
    valitse n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order mukaan n.centr_degree desc;
quit;

tiedot s17_metrics;
    aseta s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    säilytä node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
suorita;

proseduuri tulosta tiedot=s17_metrics (obs=10);
    otsikko "Section 17 — top-10 officers by degree (raw + log metrics)";
suorita;

proseduuri keskiarvot tiedot=s17_metrics n mean std min max;
    muuttuja log_degree log_pr k_val;
    otsikko "Section 17 — log-transformed metric summary";
suorita;

proseduuri princomp tiedot=s17_metrics out=s17_scores;
    muuttuja log_degree log_pr k_val;
    otsikko "Section 17 — PCA of officer network roles";
suorita;

proseduuri sgplot tiedot=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis nimike="PC1 (global influence axis)";
    yaxis nimike="PC2 (brokerage vs core embeddedness)";
    otsikko "Section 17 — officers in 2D principal-component role space";
suorita;


## 18. ARIMA-interventioanalyysi muodostumisnopeuksista

**Viite:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Sovellettu offshore-vuotoihin: Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Paradise Papers -graafin uusien yhteisöjen vuosittainen lukumäärä on
lähes monotoninen kasvusarja vuodesta 1970 (36 yhteisöä) vuoteen 2007
(1 595 yhteisöä, huippu), jota seuraa vuosien 2008-2009 pudotus ja
hitaampi tasanne vuoteen 2014 asti (vuodon kattavuuden loppu).

Sovellamme klassista Box-Tiao-interventioanalyysia testataksemme,
jättivätkö kaksi todellista tapahtumaa mitattavia jälkiä
muodostumissarjaan:

- **2009-askel** — G20:n Lontoon huippukokouksen veroparatiisien
  vastainen isku (huhtikuu 2009), joka osui yhteen finanssikriisin
  kanssa.
- **2014-askel** — Yhdysvaltain FATCA (Foreign Account Tax Compliance
  Act) tuli voimaan 1. heinäkuuta 2014.

Vaste on `log(n)`, joten interventiokerroin -0,30 vastaa noin 26 %:n
pudotusta vuosittaisessa muodostumisnopeudessa. Sovitamme
`ARIMA(1,0,0)`:n — AR(1)-autoregressiivisen mallin voimakkaasti
korreloituneelle sarjalle — kahdella askel-tekomuuttujalla
eksogeenisina `INPUT=`-muuttujina.

Nollahypoteesi on, ettei kumpikaan askel tuota mitattavaa siirtymää,
kun AR(1)-trendi on otettu huomioon. Julkaistut menetelmät eroavat
siinä, miten tulkita ei-hylkäystä: se voi tarkoittaa, ettei
interventiolla ollut vaikutusta, tai että AR(1)-autokorrelaatio
imee interventiosignaalin.


In [ ]:
proseduuri gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Rakenna interventiosarjan datajoukko. Kaksi askel-tekomuuttujaa:
   step_2009 = I{vuosi >= 2009} vangitsee G20 Lontoo/FATCA-ennakko-
   ilmoituksen regiiminmuutoksen; step_2014 = I{vuosi >= 2014}
   vangitsee FATCA:n voimaantulopäivän shokin. Vaste on log(n),
   joten kerroinestimaatit luetaan suhteellisina vaikutuksina. */
tiedot s18_ts;
    aseta year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
suorita;

proseduuri tulosta tiedot=s18_ts;
    otsikko "Section 18 — annual new-entity formations + intervention dummies";
suorita;

proseduuri sgplot tiedot=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x nimike="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x nimike="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis nimike="Incorporation year";
    yaxis nimike="New entities per year";
    otsikko "Section 18 — Paradise-Papers annual formations, 1970-2014";
suorita;

/* Tunnista malli, sitten estimoi ARIMA(1,0,0) kahdella askel-
   syötteellä. PROC ARIMAn CROSSCORR= rekisteröi eksogeeniset
   muuttujat IDENTIFY-vaiheessa, jotta ne ovat käytettävissä
   ESTIMATE INPUT=:lle. */
proseduuri arima tiedot=s18_ts;
    identify muuttuja=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimate p=1 syöte=(step_2009 step_2014);
    otsikko "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
suorita;
quit;


## 19. Nollainflatoitu pakotealtistuksen lukumäärämalli

**Viite:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2. painos, Cambridge University Press (2013), luku 4.
Nollainflatoidut mallit: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Osa 14 löysi vain **viisi** Paradise Papers -yhteisöä, joilla on
vähintään yksi toimihenkilö konsolidoidulla pakotelistalla —
häviävän harvinainen tapahtuma. Tavanomainen Poisson- tai
negatiivinen binomiregressio `sanctioned_count`:lle per yhteisö
sopisi huonosti, koska **99,98 %** yhteisöistä on nollia.
Nollainflatoitu negatiivinen binomimalli (ZINB) käsittelee tämän
jakamalla sovituksen kahteen osaan:

1. Logistinen malli sille, kuuluuko yhteisö "rakenteellisen nollan"
   luokkaan (ei mahdollista pakotealtistusta).
2. Negatiivinen binomimalli lukumäärälle loppujen joukossa.

Vain 5 positiivisen tapahtuman kanssa 21 538 yhteisössä ZINB-
uskottavuus on degeneroitunut — molemmat osat romahtavat. Tuo
epäonnistuminen on **datan rehellinen ominaisuus**, ei menettelyn.
Ajamme ZINB-sovituksen silti näyttääksemme, että regressiotyökalut
toimivat päästä päähän, ja palaamme sitten tavalliseen binääriseen
logistiseen regressioon `has_sanctioned`:lla (indikaattori
`sanctioned_count > 0`:lle). Logistinen tunnistaa selkeän
vaikutuksen: **jokainen lisä-log-yksikkö toimihenkilömäärässä
kertoo vähintään yhden pakotetun toimihenkilön kertoimet noin
3,1:llä** (p = 0,002).

Kovariaatit:

- `top5` — 6-tasoinen luokkamuuttuja (BM, KY, VG, IM, JE, OTHER),
  vertailuluokka OTHER.
- `log_n_off` — `log(officer_count)`, hallitseva kokoennustaja.


In [ ]:
/* Nouda yhteisökohtainen pakotetun toimihenkilön lukumäärä elävästä graafista. */
proseduuri gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

tiedot s19;
    aseta s19_raw;
    jos officer_count >= 1;
    pituus top5 $5;
    top5 = 'OTHER';
    jos jurisdiction = 'BM' niin top5 = 'BM';
    muuten jos jurisdiction = 'KY' niin top5 = 'KY';
    muuten jos jurisdiction = 'VG' niin top5 = 'VG';
    muuten jos jurisdiction = 'IM' niin top5 = 'IM';
    muuten jos jurisdiction = 'JE' niin top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
suorita;

proseduuri frekvenssit tiedot=s19;
    tables top5 sanctioned_count has_sanctioned;
    otsikko "Section 19 — response-variable distribution (very zero-heavy)";
suorita;

/* ZINB ensin — odotetaan degeneroituvan 5 tapahtuman sarjalla. */
proseduuri genmod tiedot=s19;
    luokka top5;
    model sanctioned_count = top5 log_n_off / dist=zinb link=log;
    otsikko "Section 19 — ZINB count model (degenerate on 5 events)";
suorita;

/* Varasija: binäärinen logistinen has_sanctioned:lla. Tulkittava. */
proseduuri logistic tiedot=s19 laskeva plots=none;
    luokka top5;
    model has_sanctioned = top5 log_n_off;
    otsikko "Section 19 — logistic regression (has-sanctioned fallback)";
suorita;


## 20. Sekavaikutus-Poisson muodostumisnopeusmalli

**Viite:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Paneelilukumäärä klassinen: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Osa 18 sovitti yksiulotteisen ARIMA:n koottuun muodostumissarjaan.
Tässä käytämme **paneeliulottuvuutta**: yksi rivi per
lainkäyttöalue-vuosi-solu, sovittaen Poisson-yleistetyn lineaarisen
sekamallin (GLMM) kiinteällä lineaarisella vuositrendillä plus
FATCA-askel-tekomuuttujalla ja **satunnaisella vakiotermillä per
lainkäyttöalue**. Tämä erottaa yhteisen trendin vaikutuksen
yksittäisen lainkäyttöalueen tasosta.

Paneelirajoitus: ne 10 lainkäyttöaluetta, joiden **keskimääräinen
vuosittainen lukumäärä** on >=5 vuosina 1990-2014 (203 solua
yhteensä). Pienemmät lainkäyttöalueet, joilla on monia nolla-
lukumäärävuosia, horjuttaisivat Poisson-sovitusta.

`PROC GLIMMIX` asetuksilla `DIST=POISSON LINK=LOG` ja
`RANDOM INTERCEPT / SUBJECT=jurisdiction` tuottaa sekä populaatio-
tason kiinteät vaikutukset (vuositrendi, FATCA-siirtymä) että
lainkäyttöalueiden välisen varianssikomponentin. Satunnaisen
vakiotermin varianssi kertoo meille, *kuinka paljon muodostumis-
nopeudet eroavat lainkäyttöalueiden välillä yhteisen aikatrendin
huomioimisen jälkeen* — rakenteellisen heterogeenisuuden piste
offshore-finanssikeskusten populaatiolle.


In [ ]:
/* Paneelidatajoukko: lainkäyttöalue x vuosi -solut vuosilta 1990-2014. */
proseduuri gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Säilytä lainkäyttöalueet, joilla keskimääräinen vuosilukumäärä >= 5. */
proseduuri sql;
    create table s20_keep_jur as
    valitse jurisdiction, avg(n) as avg_n
    from s20_raw
    group mukaan jurisdiction
    having avg(n) >= 5;
quit;

proseduuri sql;
    create table s20 as
    valitse r.*,
           r.year - 2002 as year_c,
           case kun r.year >= 2014 niin 1 muuten 0 loppu as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

proseduuri frekvenssit tiedot=s20;
    tables jurisdiction;
    otsikko "Section 20 — jurisdictions retained in the panel";
suorita;

/* Sekavaikutus-Poisson-GLMM: kiinteä vuositrendi + FATCA-askel,
   satunnainen vakiotermi lainkäyttöalueen mukaan. */
proseduuri glimmix tiedot=s20;
    luokka jurisdiction;
    model n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    otsikko "Section 20 — mixed Poisson formation-rate model";
suorita;

/* Järjestetyt satunnaiset vakiotermivaikutukset nostaisivat esiin
   lainkäyttöalueet, jotka systemaattisesti ylittävät tai alittavat
   yhteisen trendin. PROC GLIMMIXin SOLUTION-lause tulostaa nämä
   yllä olevaan oletustulosteeseen — jätämme järjestämisen lukijalle. */


In [ ]:
/* Sulje raportti-PDF ja vapauta Neo4j-kirjasto. */
ods pdf close;

libname icij clear;


## Toistettavuus ja alkuperä

- **Graafin tietolähde:** ICIJ:n *Offshore Leaks* -tutkimustietokanta,
  *Paradise Papers* -datajoukko, ensijulkaisu marraskuu 2017.
  Saatavilla osoitteessa <https://offshoreleaks.icij.org/pages/database>.
  Ladattu alustan jaettuun `step-neo4j`-palveluun (Neo4j 5.26
  Community Edition, palvelintasolla vain luku) ylävirran julkisesta
  vedoksesta osoitteessa
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **OFAC SDN -data:** Yhdysvaltain valtiovarainministeriön OFAC:n
  Specially Designated Nationals -julkinen CSV-vienti, noudettu
  valtiovarainministeriön julkisesta esikatselu-API:sta huhtikuussa
  2026. `data/ofac_sdn.csv`-tiedosto sisältää 500 kuratoitua riviä
  viidestä merkinnöiltään suurimmasta OFAC-ohjelmasta. Käytetään
  osan 6 kaksivaiheiseen seulaan.
- **OpenSanctions-data:** OpenSanctions *default* -konsolidoidun
  datajoukon tilannekuva 2026-04-17, ladattu osoitteesta
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Talletettu tiedosto `data/opensanctions_default.csv` sisältää
  18 654 Person- ja Company-skeeman riviä, joiden ensisijainen
  datajoukko on yksi OFAC SDN:n, Yhdistyneen kuningaskunnan HM
  Treasuryn, EU:n talouspakotteiden, YK:n turvallisuusneuvoston,
  Kanadan, Belgian, Australian, Sveitsin tai muiden kansallisten
  konsolidoitujen pakotelistojen joukosta. Aliakset pudotettiin,
  jotta tiedosto pysyy alle 2 Mt:n. Lisenssi: ODbL 1.0
  (OpenSanctions). Käytetään osan 14 rikastukseen.
- **Veroparatiisien sijoitukset:** Tax Justice Networkin *Corporate
  Tax Haven Index 2021* -julkaistut sijoitukset, koottu
  `https://cthi.taxjustice.net`-indeksin aloitussivulta ja maaliskuun
  2021 lehdistötiedotteesta osoitteessa
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Talletettu tiedosto `data/tax_haven_ranks.csv` luettelee ne
  lainkäyttöalueet, jotka esiintyvät Paradise Papersissa, niiden
  CTHI-sijoituksella ja johdetulla `[0, 1]`-riskipainolla. Lisenssi:
  CC BY-SA 4.0 (Tax Justice Network). Käytetään osan 15 yhdistettyyn
  pisteytykseen.
- **Graafialgoritmit:** Louvain-yhteisöjen tunnistus ja
  ominaisvektorikeskeisyys (lähin talon sisäinen vastine
  PageRankille), jotka `PROC NETWORK` laskee prosessin sisällä
  reunalistoille, jotka on noudettu vain luku -tyyppisellä
  Cypherillä. Alustan jaettu `step-neo4j`-palvelu on palvelintasolla
  vain luku, joten kaikki graafialgoritmit ajetaan työtilan podissa
  eikä Neo4j Graph Data Science -kirjoitusoperaatioiden kautta.
- **Tilastolliset menetelmät:** `PROC LIFETEST` käyttää Kaplan-
  Meier-estimaattoria log-rank-, Wilcoxon- ja Tarone-Ware-
  stratifioiduilla testeillä. `PROC PHREG` sovittaa Coxin
  suhteellisen vaarantuvuuden mallin Breslow'n sidoksin
  Python/statsmodels-kääreellä. `PROC LOGISTIC` sovittaa
  suurimman uskottavuuden binäärisen logistisen regression.
- **Riskin kokoamismenetelmät:** Osan 11 yhdistetty vaikutuspiste
  normalisoi asteen, log-PageRankin, lainkäyttöalueellisen laajuuden
  ja toimikausivuodet välille `[0, 1]` ja summaa kiinteillä painoilla
  (30/30/20/20). Osan 15 yhdistetty yhteisöjen riskipiste normalisoi
  katkaistun toimihenkilömäärän, log-PageRankin, CTHI-riskipainon ja
  binäärisen pakotetun toimihenkilön lipun ja summaa yhtäläisillä
  painoilla 0,25 kukin.

Koko analyysi on toistettavissa tämän kansion smoke-test-
skriptistä: `./smoke.jenner`. Sen ajaminen päästä päähän jaettua
`step-neo4j`-palvelua vastaan (asetettuina `JENNER_NEO4J_HOST` ja
`JENNER_NEO4J_PASS`, jotka alusta asettaa puolestasi missä tahansa
työtilan podissa) kestää noin kaksi minuuttia ja varmistaa, että
jokainen kysely ja jokainen PROC — mukaan lukien viisi uutta osaa,
jotka lisättiin olemassa olevan putken rinnalle — palauttaa odotetun
tulosteen todellisella 163 435 solmun graafilla.
